# 21 - ReAct / Action-Observation Loop

ReAct is commonly explained as **reasoning + acting**.

For engineering, the important observable pattern is:

```text
decide
  ↓
act
  ↓
observe
  ↓
decide again
```

The model may use tools, inspect results, and continue until the task is complete.

## Setup

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

CHAT_MODEL = os.getenv("OPENAI_CHAT_MODEL", "")
if CHAT_MODEL:
    print("Configured model:", CHAT_MODEL)
else:
    print("Set OPENAI_CHAT_MODEL for live model examples.")

## 1. Manual action-observation loop

This educational loop does not expose hidden chain-of-thought. It models the observable action/tool/result cycle.

In [ ]:
TOOLS = {
    'calculator': lambda expression: eval(
        expression, {'__builtins__': {}}, {}
    )
}

history = []

actions = [
    {'tool': 'calculator', 'args': {'expression': '25 * 4'}},
]

for action in actions:
    tool = TOOLS[action['tool']]
    observation = tool(**action['args'])
    history.append({
        'action': action,
        'observation': observation,
    })

print(history)

## 2. ReAct state

A useful state can record:
- user goal
- actions taken
- observations
- current status

Do not treat private model reasoning as an application log. Log the actions and observations your system is allowed to record.

In [ ]:
react_state = {
    'goal': 'calculate total',
    'actions': [
        {'tool': 'calculator', 'args': {'expression': '25 * 4'}}
    ],
    'observations': [100],
    'status': 'ready_for_answer',
}

print(react_state)

## 3. Tool failure and recovery

Observation can be:
- normal result
- error
- timeout
- empty result

The next decision can change accordingly.

In [ ]:
def run_action(tool_name, expression):
    try:
        result = TOOLS[tool_name](expression)
        return {'ok': True, 'result': result}
    except Exception as exc:
        return {'ok': False, 'error': str(exc)}

print(run_action('calculator', '25 * 4'))
print(run_action('calculator', 'bad + input'))

## 4. Why the loop matters

The loop is the bridge from one-shot generation to agentic behavior.

```text
one-shot model → answer

agent:
model → action → observation → model → ... → answer
```

## 5. Safety and stopping

Add:
- max steps
- timeout
- allowed tools
- action validation
- duplicate-action detection
- budget

In [ ]:
MAX_STEPS = 6

def should_stop(step, status):
    return step >= MAX_STEPS or status == 'ready_for_answer'

print(should_stop(2, 'working'))
print(should_stop(6, 'working'))

## 6. Evaluation and practice

Evaluate:
- successful trajectory
- unnecessary actions
- failure recovery
- stop correctness
- tool choice

Practice: create a two-tool task requiring a sequence of actions.

## Evaluation

The trajectory matters. A final correct answer can still hide a poor agent trajectory.

Improve with:
- better tool descriptions
- planning when useful
- step budgets
- verifier nodes
- observability.

## References / current documentation


- LangChain reference: https://reference.langchain.com/python/langchain/overview
- LangGraph reference: https://reference.langchain.com/python/langgraph/overview
- LangGraph persistence: https://github.com/langchain-ai/docs/blob/main/src/oss/langgraph/persistence.mdx
- LangGraph interrupts: https://github.com/langchain-ai/docs/blob/main/src/oss/langgraph/interrupts.mdx
- LangChain human-in-the-loop: https://github.com/langchain-ai/docs/blob/main/src/oss/langchain/human-in-the-loop.mdx
- Ragas metrics: https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/
- MCP Python SDK v2: https://py.sdk.modelcontextprotocol.io/
- MCP Python SDK getting started: https://py.sdk.modelcontextprotocol.io/get-started/

Your supplied agent notebook uses the same decision/action/observation loop as the core agent idea. fileciteturn19file0